In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset, Dataset
import os
from pathlib import Path
import torch

In [ ]:
os.environ["HF_CACHE"] = "/sadata/nikkothari/cache/hf/hub/"

OUTPUT_DIR = "/sadata/nikkothari/kai-scheduler-lab/train/ckpts/"
MODEL_NAME = "Qwen/Qwen3-0.6B"
TRAIN_DATA_PATH = "/sadata/nikkothari/kai-scheduler-lab/train/data/train.json"
EVAL_DATA_PATH = "/sadata/nikkothari/kai-scheduler-lab/train/data/eval.json"
MAX_LENGTH = 4096

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

In [ ]:
model.device

In [ ]:
model

In [ ]:
train_dataset = load_dataset("trl-lib/Capybara", split="train")
test_dataset = load_dataset("trl-lib/Capybara", split="test")
train_dataset[0], test_dataset[0]

In [ ]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

### First Way

In [ ]:
collator(
    [{"input_ids": [32, 31, 32], "attention_mask": [1,1,1]},
     {"input_ids": [32, 31, 32, 2,2,2,2,2,2,2,2,2,2], "attention_mask": [1,1,1,1,1,1,1,1,1,1,1,1,1]}]
)

### Second Way

In [ ]:
messages = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a short joke about saving RAM."},
        {"role": "assistant", "content": "RAM.. RAM.."}
    ],
    [
        {"role": "user", "content": "Write a short joke."},
        {"role": "assistant", "content": "RANDOM.."}
    ]
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False,                # Padding happens in the collator
    add_generation_prompt=False,  # These conversations include the answers
)

features = [
    {key: values[i] for key, values in inputs.items()}
    for i in range(len(inputs["input_ids"]))
]

batch = collator(features)

for key, value in batch.items():
    print(key, value.shape)
    print(value)

### Final Way

In [ ]:
tokenize_function = lambda messages: tokenizer.apply_chat_template(
    messages["messages"],
    tokenize=True,
    return_dict=True,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False,
    add_generation_prompt=False,
)

remove_columns = list(train_dataset[0].keys())
tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=remove_columns)
tokenized_dataset = list(tokenized_dataset)

In [ ]:
batch = collator(tokenized_dataset[:2])

In [ ]:
for key, value in batch.items():
    print(key, value.shape)
    print(value)

In [ ]:
get_last_checkpoint("/sadata/nikkothari/kai-scheduler-lab/train/ckpts")